# Text-to-text LLM Evaluation

## Import packages

In [14]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import json
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass
from pathlib import Path

from langchain_ollama.llms import OllamaLLM as Ollama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from neurorag.models.OpenRouter import OpenRouter
import transformers
import torch

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  bert_score_metric,
)

## Disable warnings

In [15]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

**Note:** FActScore metric requires an OpenAI API key to function properly, as it uses GPT models for fact extraction and verification.

## Import packages

In [16]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Define evaluation function

In [17]:
def eval_rag(chain) -> float:
  dataset_df = pd.read_csv('../datasets/mediqa.csv')
  expected_answers = dataset_df['answer']
  predicted_answers = []

  for index, row in tqdm(list(dataset_df.iterrows()), desc='Questions'):
    question = row['question']
    llm_answer = chain.invoke({'query': question})
    predicted_answers.append(llm_answer)

  cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
  factscore = factscore_metric(expected_answers, predicted_answers)
  bert_score = bert_score_metric(expected_answers, predicted_answers)

  return cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore, bert_score

## Define prompt

In [18]:
template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
Keep the answer verbose, with a minimum of three paragraphs.

QUERY: {query}

First, identify the key scientific concepts and data points in the CONTEXT that relate to the QUERY.
Then, analyze how these concepts connect to form a comprehensive answer.
Finally, synthesize your findings into a detailed response.
"""

prompt = PromptTemplate(
  template=template,
  input_variables=['query'],
)

## Setup LLMs

### Llama 3.3 70B Instruct

In [19]:
def get_llama3_3_70b_instruct_llm(temperature=0.0):
  return OpenRouter(model='meta-llama/llama-3.3-70b-instruct', temperature=temperature)

### Mistral Large

In [20]:
def get_mistral_large_llm(temperature=0.0):
  return OpenRouter(
    model='mistralai/mistral-large',
    temperature=temperature,
  )

### GPT-4.1

In [21]:
def get_chatgpt_4_1_llm(temperature=0.0):
  return OpenRouter(
    model='openai/gpt-4.1',
    temperature=temperature,
  )

### OpenBioLLM 70B Q2_k

In [22]:
def get_openbiollm_70_Q2k_llm(temperature=0.0):
  return Ollama(model='taozhiyuai/openbiollm-llama-3:70b_q2_k', temperature=temperature)

### Biomistral 7B Q4_k_m

In [23]:
def get_biomistral_q4_k_m_llm(temperature=0.0):
  return Ollama(model='cniongolo/biomistral', temperature=temperature)

### OpenBioLLM Llama3 70B (HuggingFace)

In [24]:
def get_openbiollm_llama3_70b_llm(temperature=0.0):
  model_id = "aaditya/OpenBioLLM-Llama3-70B"
  
  pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
  )
  
  # Configure the pipeline with terminators and generation parameters
  terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
  ]
  
  # Create HuggingFace pipeline wrapper for LangChain
  hf_pipeline = HuggingFacePipeline(
    pipeline=pipeline,
    pipeline_kwargs={
      "max_new_tokens": 512,
      "eos_token_id": terminators,
      "do_sample": True if temperature > 0 else False,
      "temperature": temperature if temperature > 0 else None,
      "top_p": 0.9,
    }
  )
  
  return hf_pipeline

## Evaluate the models

### Load QA dataset

In [25]:
mediqa_df = pd.read_csv('../datasets/pubmed_summary_qa.csv')[:50]
mediqa_df

,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...
5,How does the auditory system respond to differ...,The auditory system's response to sound varies...
6,What is tonotopic organization in the auditory...,Tonotopic organization refers to the mapping o...
7,What brain regions are involved in language pr...,Language processing involves areas in the pref...
8,How does bilingualism affect language processi...,Bilingualism is associated with overlapping ac...
9,What is functional magnetic resonance imaging ...,Functional magnetic resonance imaging (fMRI) i...


### Setup experiment grid search parameters

In [26]:
llms = (
  ('Llama 3.3 70B Instruct', get_llama3_3_70b_instruct_llm()),
  ('GPT-4.1', get_chatgpt_4_1_llm()),
  ('Mistral Large', get_mistral_large_llm()),
  # ('OpenBioLLM Llama3 70B', get_openbiollm_llama3_70b_llm()),
  # ('Biomistral 7B Q4_k_m', get_biomistral_q4_k_m_llm()),
)

### Load cached RAGs responses

In [27]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-llm-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache.keys())

2

### Conduct the grid search

In [28]:
df = pd.DataFrame()

questions = mediqa_df['question'].tolist()
expected_answers = mediqa_df['answer'].tolist()

for llm_name, llm in llms:
  chain = prompt | llm | StrOutputParser()

  predicted_answers = []

  if llm_name not in cache[CACHE_KEY]:
    cache[CACHE_KEY][llm_name] = {}

  for question in tqdm(questions, desc='Questions'):
    if question not in cache[CACHE_KEY][llm_name]:
      cache[CACHE_KEY][llm_name][question] = chain.invoke(question)

    predicted_answers.append(cache[CACHE_KEY][llm_name][question])

    with open(cache_path, 'w') as f:
      json.dump(cache, f)

  # Evaluate metrics
  cos_sim = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
  factscore = factscore_metric(expected_answers, predicted_answers)
  bert_score = bert_score_metric(expected_answers, predicted_answers)

  # Save results
  row = pd.DataFrame({
    'llm': llm_name,
    'cos_sim': cos_sim,
    'bleu': bleu_score,
    'rogue_1': rogue_1_score,
    'rogue_l': rogue_l_score,
    'factscore': factscore,
    'bert_score': bert_score,
  }, index=[0])
  df = pd.concat([df, row], ignore_index=True)

df.sort_values(by='cos_sim', ascending=False)

Questions: 100%|██████████| 50/50 [27:10<00:00, 32.62s/it]


,llm,cos_sim,bleu,rogue_1,rogue_l,factscore,bert_score
1,GPT-4.1,0.742768,0.010947,0.313494,0.253310,0.377358,0.039305
2,Mistral Large,0.723717,0.005425,0.314345,0.255528,0.188313,0.090085
0,Llama 3.3 70B Instruct,0.722774,0.010823,0.305830,0.242718,0.375027,0.105946
